# Sentralitet

Noen noder er alltid mer sentrale enn andre.  En nærmere undersøkelse viser at det er mange måter å "dfinere" det å være sentral, og alle har fordeler (og ulemper).
De viktigste (_no pun intended_) er 
- Populære noder (_degree centrality_)
- Bronoder (_betweenness centrality_)
- Sentrale noder (_closeness centrality_)
- Viktige noder (_page rank_)

Vi skal se på alle fire


|Type|Hva som måles|Idé|Observasjon|
|---|---|---|---|
|Popularitet|Antall kanter|Hvor mange venner har du|Tar ikke hensyn til "kvaliteten" på vennene dine|
|Bronoder|Antall korteste (andres)<br>sti gjennom noden|Knytter sammen samfunnet,<br>informasjon flyter hit raskt|Du kan være en bro uten å være viktig selv|
|Sentrale|Antall korteste sti til andre noder|Nær der ting skjer|Følsom for endringer - tung å beregne|
|Viktig|Aggregert viktighet av naboene|Er viktig ved å kjenne mange viktige|Må iterere over hele grafen|



## Hente inn epost-grafen

In [1]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import gzip

G = nx.DiGraph()
# husk at gzip påpner i 'b'
with gzip.open("data/email.edgelist.txt.gz", "rt") as fd:
    for linje in fd:
         link = linje.split()
         G.add_edge(int(link[0]), int(link[1]))
    #
#
print(f"Antall noder i grafen: {G.number_of_nodes()}")
print(f"Antall kanter i grafen: {G.number_of_edges()} (antall eposter)")
# Bort med eposter sendt til seg selv
G.remove_edges_from(nx.selfloop_edges(G))
print(f"Antall kanter etter selv-loop: {G.number_of_edges()}")
# Og noder uten linker til seg nå
isolerte = list(nx.isolates(G)) # kan ikke bruke iteratorer direkte
G.remove_nodes_from(isolerte)
print(f"Antall noder etter isolerte: {G.number_of_nodes()}")
# ANtall kanter i hver node
antall = []
for node, naboer in G.degree():
    antall += [naboer]
    # eller antall = [d for n, d in G.degree()]
#
print(f"Funnet Stdev:   {np.std(antall):.4f}")
print(f"Funnet Median:   {np.mean(antall):.4f}")

Antall noder i grafen: 57194
Antall kanter i grafen: 103731 (antall eposter)
Antall kanter etter selv-loop: 103083
Antall noder etter isolerte: 57189
Funnet Stdev:   36.0183
Funnet Median:   3.6050


La oss se på noen tilfeldige noder (håper at statistikken ikke gir oss et puss her):

In [2]:
import random

for i in range(10):
    node = int(random.random() * G.number_of_nodes())
    print(f"Node {node}: {len(G.in_edges(node))} {len(G.out_edges(node))}")

Node 1769: 11 0
Node 17820: 0 1
Node 29526: 1 1
Node 10084: 0 1
Node 5869: 1 0
Node 23193: 1 1
Node 39002: 0 1
Node 28440: 1 0
Node 28786: 2 2
Node 27478: 0 1


Men la oss se på noen utvalgte noder

In [3]:
print(f"63    ut: {len(G.out_edges(63))}  in: {len(G.in_edges(63))}")
print(f"40    ut: {len(G.out_edges(40))}   in: {len(G.in_edges(40))}")
print(f"407   ut: {len(G.out_edges(407))}   in: {len(G.in_edges(407))}")
print(f"1704  ut: {len(G.out_edges(1704))}  in: {len(G.in_edges(1704))}")
print(f"11798 ut: {len(G.out_edges(11798))} in: {len(G.in_edges(11798))}")
print(f"11028 ut: {len(G.out_edges(11028))} in: {len(G.in_edges(11028))}")
print(f"32199 ut: {len(G.out_edges(32199))} in: {len(G.in_edges(32199))}")

63    ut: 13  in: 29
40    ut: 0   in: 274
407   ut: 0   in: 205
1704  ut: 51  in: 107
11798 ut: 881 in: 262
11028 ut: 4170 in: 8
32199 ut: 6553 in: 2


## Populære noder
Dette er det enkleste: Antall linker (til andre noder).  Kjører i $O(N)$ som en følge av at man må traversere alle noder (og vi husker at fordi sortering "bare" er $O(\log N)$ er totalen $O(N)$ og ikke $O(N) + O(N \log N)$).

Sier ikke så mye om hvorvidt noden er relevant eller ei.  Kan godt være at jeg kjenner flere i PIT enn Direktøren, men de hun kjenner er viktigere.

In [4]:
import time

start = time.time()
populæritet = nx.degree_centrality(G)
top_populære_noder = sorted(populæritet, key=populæritet.get, reverse=True)[:5]
print("5 mest populære:")
for n in top_populære_noder:
    print(f"\t Node: {n} Naboer: {G.degree[n]}")
#
print(f"Å finne de fem mest populære tok {time.time()-start:.2f} sekunder")

5 mest populære:
	 Node: 32199 Naboer: 6555
	 Node: 11028 Naboer: 4178
	 Node: 13678 Naboer: 1228
	 Node: 11798 Naboer: 1143
	 Node: 13498 Naboer: 809
Å finne de fem mest populære tok 0.02 sekunder


Legg merke til at det her telles naboer uten å ta hensyn til retningen.

## Bronoder
En bro som knytter en øy til fastlandet, den er viktig fordi alle må over den.  Broen har bare to kanter til seg (én på hver side) og ikke populær (se over), og broen behøver ikke være nær noe viktig (sentrum av byen) men **posisjonen** relativt til andre noder i grafen gjør den viktig.  En bronode beskriver en strukturell plassering, heller enn egenskaper ved noden selv.

Teknisk er broen på korteste sti mellom mange andre (uansett hvem du skal besøke må du over broen).

Det følger av posisjonen at om en bronode feiler deles grafen.  Man finner bronoder ved å se hva som skjer når noder fjernes; de nodene som fører til at grafen separeres, de er bronoder.  Med andre ord: Ved å fjerne (fengsle?) bronoder fragmenterer man grafen.

Å finne bronoder krever å finne korteste sti for alle noder.  Raskeste algoritme er [Brande's algoritme](https://en.wikipedia.org/wiki/Brandes%27_algorithm) som kjører i $O(VE)$ tid.  Det betyr at kompleksiteten stiger lineært både med hvor "tett" den er og hvor "stor" den er.

Fordi bronoder står mellom grupper av andre norder er den engelske termen _betweenness_.

In [ ]:
import time

start = time.time()
broer = nx.betweenness_centrality(G)
top_bronoder = sorted(broer, key=broer.get, reverse=True)[:5]
print("5 viktigste bronoder")
for n in top_bronoder:
    print(f"\t Node: {n} Naboer: {G.degree[n]}")
#
print(f"Å finne bronoder tok {time.time()-start:.2f} sekunder")


I en rettet graf må man være nøye med å definere hva en deling av grafen betyr (og derfor hva en bronode er).  Særlig: En node kan forbinde to deler i én retning men ikke den andre.

## Sentrale noder
Hvor tilgjengelig noden er for andre (lav gjennomsnitt korteste sti til andre noder).  Direktøren i PIT er trolig mest sentral i organisasjonen, som en følge av at hun har kortere vei "ned" til alle ansatte enn noen andre.

Sentrale noder har kort vei til andre, og vil derfor høre rykter hurtig.  Og, vice versa, kunne spre rykter og nyheter raskt.

Om vi antar at kantene hverken har retning eller vekt, da bruker vi bredde-først og kompleksiteten er $O(N^2)$ men om antall kanter er høyt nærmer kompleksiteten seg $O(N^3$) (hvor $N$ er antall noder).  Det skal ikke så mye til før det er ønskelig med tilnærminger heller enn eksakte løsninger.

Det er en likhet mellom bronoder og sentrale noder.

Fordi kort vei tli andre er essensen, heter dise nodene _closeness_ på engelsk.

In [ ]:
start = time.time()
sentrale = nx.closeness_centrality(UG)
top_sentrale = sorted(sentrale, key=sentrale.get, reverse=True)[:5]
print("5 mest sentrale noder:")
for n in top_sentrale:
    print(f"\t Node: {n} Naboer: {UG.degree[n]}")
#   
print(f"Å finne sentrale noder {time.time()-start:.2f} sekunder")

## Viktige noder
Om vi myser litt, ser vi at populære, sentrale, og bronoder alle har sine egenskaper som en følge av sin plassering i grafen.  Det vil si,  


In [ ]:
# Beregne de viktigste nodene
page_rank = nx.eigenvector_centrality(G)

viktigste_noder = sorted(page_rank, key=page_rank.get, reverse=True)[:5]

print("5 mest populære:")
for n in viktigste_noder:
    print(f"\t Node {n}  har {G.degree[n]} naboer")
#   